# EEL204 Assignment 2: Z0 of three transmission lines by the finite difference method 
# Lakshya Soni 
# B25EE027
We put the strip at 1 V and every ground at 0 V, solve Laplace's equation on the cross-section,
and find the charge on the strip. That gives the capacitance per metre:
$C$ with the real dielectric and $C_0$ with air everywhere. Then

$$Z_0 = \frac{1}{c\sqrt{C\,C_0}}, \qquad \epsilon_{\text{eff}} = \frac{C}{C_0}.$$

The stripline has only one dielectric, so $C = \epsilon_r C_0$ and one solve is enough: $Z_0 = \sqrt{\epsilon_r}/(c\,C)$.

All three lines are symmetric, so we solve only the right half. Column 0 of every array is the centre line.

In [ ]:
import os
import time
import numpy as np
import matplotlib.pyplot as plt

os.makedirs("figures", exist_ok=True)

speed_of_light = 299_792_458.0          # m/s
vacuum_permittivity = 8.8541878128e-12  # F/m
tolerance = 1e-6                        # stop when no point changes by more than this (V)

## The three lines

Each line is a short list of settings. `strip_height_mm` is how far the strip is above the bottom ground.
`box_width_mm` is the distance from the centre to the side wall, and `box_height_mm` is the space above the strip.

In [ ]:
microstrip = dict(relative_permittivity=4.1, strip_height_mm=0.9, strip_width_mm=1.2,
                  box_width_mm=20, box_height_mm=20, filled=False, top_ground_from_mm=None,
                  margin=4, steps=[0.1, 0.05, 0.025])

# h = 0.9 mm from the strip to each plate, so the plates are 1.8 mm apart
stripline = dict(relative_permittivity=9, strip_height_mm=0.9, strip_width_mm=1.2,
                 box_width_mm=8, box_height_mm=0.9, filled=True, top_ground_from_mm=None,
                 margin=4, steps=[0.05, 0.025, 0.0125])

# top grounds start W/2 + S from the centre; the Gauss rectangle must fit in the 0.3 mm gap
gcpw = dict(relative_permittivity=2.5, strip_height_mm=1.3, strip_width_mm=0.8,
            box_width_mm=10, box_height_mm=10, filled=False, top_ground_from_mm=0.8 / 2 + 0.3,
            margin=2, steps=[0.05, 0.025, 0.0125])

## Grid

Row 0 is the bottom ground. The strip lies on row `strip_row`. The permittivity is stored on the links
between points: a link below the strip row is in the dielectric, a link above it is in air, and a sideways
link on the strip row is half and half, so it gets $(\epsilon_r + 1)/2$.

In [ ]:
def make_grid(line, dx, relative_permittivity):
    strip_row = round(line["strip_height_mm"] / dx)
    strip_end = round(line["strip_width_mm"] / 2 / dx)
    rows = strip_row + round(line["box_height_mm"] / dx) + 1
    columns = round(line["box_width_mm"] / dx) + 1

    V = np.zeros((rows, columns))
    fixed = np.zeros((rows, columns), dtype=bool)
    fixed[0, :] = fixed[-1, :] = fixed[:, -1] = True     # grounded box, 0 V
    V[strip_row, :strip_end + 1] = 1.0                   # strip, 1 V
    fixed[strip_row, :strip_end + 1] = True
    if line["top_ground_from_mm"]:                       # GCPW top grounds, 0 V
        fixed[strip_row, round(line["top_ground_from_mm"] / dx):] = True

    top = relative_permittivity if line["filled"] else 1.0   # what is above the strip
    below = np.arange(rows) < strip_row
    vertical = np.where(below[:-1], relative_permittivity, top)
    horizontal = np.where(below, relative_permittivity, top)
    horizontal[strip_row] = (relative_permittivity + top) / 2
    return dict(V=V, fixed=fixed, vertical=vertical, horizontal=horizontal,
                strip_row=strip_row, strip_end=strip_end)

## Solver: successive over-relaxation

Each free point moves towards the weighted average of its four neighbours, with the link permittivities
as weights. Inside one material this is the plain average. On the air/dielectric row it is the formula
from Iskander et al.:

$$V = \frac{V_{\text{up}} + \epsilon_r V_{\text{down}} + \tfrac{1+\epsilon_r}{2}(V_{\text{left}} + V_{\text{right}})}{2(1+\epsilon_r)}$$

Each point is pushed a little past the average: $V \leftarrow V + \omega(V_{\text{avg}} - V)$, with
$\omega = 2/(1+\sin(\pi/N))$. Points are coloured like a chessboard; all red points are updated at once,
then all black ones. At the centre line the missing left neighbour is a copy of the right one.

In [ ]:
def solve(grid, start=None):
    fixed = grid["fixed"]
    V = grid["V"].copy() if start is None else np.where(fixed, grid["V"], start)
    omega = 2 / (1 + np.sin(np.pi / max(V.shape)))

    up = grid["vertical"][1:][:, None]
    down = grid["vertical"][:-1][:, None]
    side = grid["horizontal"][1:-1][:, None]
    total = up + down + 2 * side

    free = ~fixed[1:-1, :-1]
    i, j = np.indices(free.shape)
    red = free & ((i + j) % 2 == 0)
    black = free & ((i + j) % 2 == 1)

    for iteration in range(1, 50_001):
        largest_change = 0.0
        for colour in (red, black):
            centre = V[1:-1, :-1]                       # a view: writing to it changes V
            left = np.concatenate([V[1:-1, 1:2], V[1:-1, :-2]], axis=1)
            average = (up * V[2:, :-1] + down * V[:-2, :-1] + side * (left + V[1:-1, 1:])) / total
            change = omega * (average - centre)
            centre[colour] += change[colour]
            largest_change = max(largest_change, np.abs(change[colour]).max())
        if largest_change < tolerance:
            return V, iteration
    raise RuntimeError("SOR did not converge")

## Charge by Gauss's law

A rectangle around the strip, halfway between points. Each link that crosses it adds
$\epsilon_{\text{link}}(V_{\text{in}} - V_{\text{out}})$; the grid step cancels because $\Delta x = \Delta y$.
The point on the centre line owns only half its segment. The sum is doubled for the left half.

In [ ]:
def charge(V, grid, margin):
    top = grid["strip_row"] + margin
    bottom = grid["strip_row"] - margin
    right = grid["strip_end"] + margin
    vertical, horizontal = grid["vertical"], grid["horizontal"]
    share = np.ones(right + 1)
    share[0] = 0.5

    through_top = vertical[top] * (V[top, :right + 1] - V[top + 1, :right + 1])
    through_bottom = vertical[bottom - 1] * (V[bottom, :right + 1] - V[bottom - 1, :right + 1])
    through_side = horizontal[bottom:top + 1] * (V[bottom:top + 1, right] - V[bottom:top + 1, right + 1])
    flux = (share * through_top).sum() + (share * through_bottom).sum() + through_side.sum()
    return 2 * vacuum_permittivity * flux

## Z0 on one grid, and on three grids

A finer grid starts from the coarser answer, doubled in resolution (`refine`); this saves most of the work.
Halving the step halves the error, so with three grids we can estimate the answer for $\Delta \to 0$
(Richardson extrapolation):

$$p = \log_2\frac{Z_1 - Z_2}{Z_2 - Z_3}, \qquad Z_0 \approx Z_3 + \frac{Z_3 - Z_2}{2^p - 1}$$

In [ ]:
def refine(V, shape):
    fine = np.zeros(shape)
    fine[::2, ::2] = V
    fine[1::2, ::2] = (fine[:-1:2, ::2] + fine[2::2, ::2]) / 2
    fine[:, 1::2] = (fine[:, :-1:2] + fine[:, 2::2]) / 2
    return fine


def impedance(line, dx, previous=None, **changes):
    line = {**line, **changes}
    start = time.perf_counter()
    er = line["relative_permittivity"]

    grid = make_grid(line, dx, er)
    V, iterations = solve(grid, None if previous is None else refine(previous["V"], grid["V"].shape))
    C = charge(V, grid, line["margin"])
    result = dict(dx=dx, V=V, shape=V.shape)

    if line["filled"]:                  # one dielectric: C = er * C0
        result.update(Z0=np.sqrt(er) / (speed_of_light * C), effective_permittivity=er)
    else:                               # solve again with air everywhere
        air = make_grid(line, dx, 1.0)
        V_air, more = solve(air, None if previous is None else refine(previous["V_air"], air["V"].shape))
        C0 = charge(V_air, air, line["margin"])
        iterations += more
        result.update(V_air=V_air, Z0=1 / (speed_of_light * np.sqrt(C * C0)), effective_permittivity=C / C0)

    result.update(iterations=iterations, seconds=time.perf_counter() - start)
    return result


def three_grids(line):
    results = []
    for dx in line["steps"]:
        r = impedance(line, dx, results[-1] if results else None)
        results.append(r)
        print(f"dx = {dx:<7} mm  grid {r['shape'][0]} x {r['shape'][1]}  iterations {r['iterations']:>5}  "
              f"time {r['seconds']:6.1f} s  Z0 = {r['Z0']:.3f} ohm  eff. permittivity = {r['effective_permittivity']:.4f}")

    Z1, Z2, Z3 = (r["Z0"] for r in results)
    p = np.log2((Z1 - Z2) / (Z2 - Z3))
    Z0 = Z3 + (Z3 - Z2) / (2 ** p - 1)
    e2, e3 = results[1]["effective_permittivity"], results[2]["effective_permittivity"]
    effective_permittivity = e3 + (e3 - e2) / (2 ** p - 1)
    print(f"order p = {p:.2f}   Z0 extrapolated = {Z0:.2f} ohm   eff. permittivity = {effective_permittivity:.3f}   "
          f"total time {sum(r['seconds'] for r in results):.1f} s")
    return results, Z0, effective_permittivity

## Reference formulas (independent check, no FDM)

- Microstrip: Hammerstad and Jensen (1980), accurate to 0.2 %.
- Stripline: Cohn (1954), exact for a zero-thickness strip.
- GCPW: conformal mapping (Ghione and Naldi, 1987), good to a few per cent.

$K$ is the complete elliptic integral, computed with the arithmetic-geometric mean.

In [ ]:
def K(k):
    a, b = 1.0, np.sqrt(1 - k**2)
    while abs(a - b) > 1e-15:
        a, b = (a + b) / 2, np.sqrt(a * b)
    return np.pi / (2 * a)


def K_ratio(k):
    return K(k) / K(np.sqrt(1 - k**2))


def hammerstad_jensen(er, h, W):
    u = W / h
    f = 6 + (2 * np.pi - 6) * np.exp(-(30.666 / u) ** 0.7528)
    Z0_air = 376.730313668 / (2 * np.pi) * np.log(f / u + np.sqrt(1 + (2 / u) ** 2))
    a = 1 + np.log((u**4 + (u / 52) ** 2) / (u**4 + 0.432)) / 49 + np.log(1 + (u / 18.1) ** 3) / 18.7
    b = 0.564 * ((er - 0.9) / (er + 3)) ** 0.053
    effective = (er + 1) / 2 + (er - 1) / 2 * (1 + 10 / u) ** (-a * b)
    return Z0_air / np.sqrt(effective)


def cohn_stripline(er, plate_spacing, W):
    return 30 * np.pi / np.sqrt(er) * K_ratio(1 / np.cosh(np.pi * W / (2 * plate_spacing)))


def conformal_gcpw(er, h, W, S):
    a, b = W / 2, W / 2 + S
    k, k3 = a / b, np.tanh(np.pi * a / (2 * h)) / np.tanh(np.pi * b / (2 * h))
    q = K_ratio(k3) / K_ratio(k)
    effective = (1 + er * q) / (1 + q)
    return 60 * np.pi / np.sqrt(effective) / (K_ratio(k) + K_ratio(k3))


reference = {"microstrip": hammerstad_jensen(4.1, 0.9, 1.2),
             "stripline": cohn_stripline(9, 1.8, 1.2),
             "GCPW": conformal_gcpw(2.5, 1.3, 0.8, 0.3)}
for name, value in reference.items():
    print(f"{name:<10} reference Z0 = {value:.2f} ohm")

## Potential plot

In [ ]:
def plot_potential(result, line, title, file_name, x_limit_mm, y_top_mm):
    V, dx = result["V"], result["dx"]
    full = np.concatenate([V[:, :0:-1], V], axis=1)          # mirror the right half
    x = np.arange(-(V.shape[1] - 1), V.shape[1]) * dx
    y = np.arange(V.shape[0]) * dx
    strip_y, half = line["strip_height_mm"], line["strip_width_mm"] / 2

    figure, axes = plt.subplots(figsize=(8, 3.6))
    shown = axes.pcolormesh(x, y, full, cmap="viridis", shading="gouraud", vmin=0, vmax=1)
    axes.contour(x, y, full, levels=np.linspace(0.1, 0.9, 9), colors="white", linewidths=0.4, alpha=0.7)
    if not line["filled"]:
        axes.axhline(strip_y, color="orange", linestyle="--", linewidth=1, label="substrate top")
    axes.plot([-half, half], [strip_y] * 2, color="red", linewidth=3, label="strip (1 V)")
    if line["top_ground_from_mm"]:
        start = line["top_ground_from_mm"]
        axes.plot([start, x_limit_mm], [strip_y] * 2, color="black", linewidth=3, label="top ground (0 V)")
        axes.plot([-x_limit_mm, -start], [strip_y] * 2, color="black", linewidth=3)
    axes.set(xlim=(-x_limit_mm, x_limit_mm), ylim=(0, y_top_mm), aspect="equal",
             xlabel="x (mm)", ylabel="y (mm)", title=title)
    axes.legend(loc="upper right", fontsize=7)
    figure.colorbar(shown, ax=axes, label="potential (V)", shrink=0.9)
    figure.tight_layout()
    figure.savefig(f"figures/{file_name}", dpi=200)
    plt.show()

# (a) Microstrip: $\epsilon_r = 4.1$, $h = 0.9$ mm, $W = 1.2$ mm

In [ ]:
microstrip_results, microstrip_Z0, microstrip_effective = three_grids(microstrip)
plot_potential(microstrip_results[-1], microstrip, "(a) Microstrip potential, dx = 0.025 mm",
               "microstrip_potential.png", x_limit_mm=5, y_top_mm=4)

Box size check on the 0.1 mm grid (the real line has open air above it):

In [ ]:
for size in [5, 10, 15, 20, 30]:
    print(f"box {size:>2} mm: Z0 = {impedance(microstrip, 0.1, box_width_mm=size, box_height_mm=size)['Z0']:.3f} ohm")

# (b) Stripline: $\epsilon_r = 9$, $h = 0.9$ mm (plates 1.8 mm apart), $W = 1.2$ mm

In [ ]:
stripline_results, stripline_Z0, stripline_effective = three_grids(stripline)
plot_potential(stripline_results[-1], stripline, "(b) Stripline potential, dx = 0.0125 mm",
               "stripline_potential.png", x_limit_mm=3, y_top_mm=1.8)

Side wall check on the 0.05 mm grid (the real stripline has no side walls):

In [ ]:
for distance in [2, 3, 4, 6, 8, 12]:
    print(f"side wall at {distance:>2} mm: Z0 = {impedance(stripline, 0.05, box_width_mm=distance)['Z0']:.4f} ohm")

# (c) GCPW: $\epsilon_r = 2.5$, $h = 1.3$ mm, $W = 0.8$ mm, $S = 0.3$ mm

In [ ]:
gcpw_results, gcpw_Z0, gcpw_effective = three_grids(gcpw)
plot_potential(gcpw_results[-1], gcpw, "(c) GCPW potential, dx = 0.0125 mm",
               "gcpw_potential.png", x_limit_mm=3, y_top_mm=3)

Box size check on the 0.05 mm grid:

In [ ]:
for size in [2, 4, 6, 10, 15]:
    print(f"box {size:>2} mm: Z0 = {impedance(gcpw, 0.05, box_width_mm=size, box_height_mm=size)['Z0']:.3f} ohm")

# All three lines

In [ ]:
lines = [("microstrip", microstrip_results, microstrip_Z0, microstrip_effective),
         ("stripline", stripline_results, stripline_Z0, stripline_effective),
         ("GCPW", gcpw_results, gcpw_Z0, gcpw_effective)]

figure, axes = plt.subplots(figsize=(6, 3.5))
for name, results, Z0, _ in lines:
    axes.plot([r["dx"] for r in results], [100 * (r["Z0"] - Z0) / Z0 for r in results], "o-", label=name)
axes.axhline(0, color="grey", linewidth=0.8)
axes.set(xlabel="grid step dx (mm)", ylabel="Z0 error vs extrapolated value (%)",
         title="Grid convergence: the error halves when dx halves")
axes.legend()
figure.tight_layout()
figure.savefig("figures/all_convergence.png", dpi=200)
plt.show()

print(f"{'line':<11}{'Z0 finest':>11}{'Z0 extrap.':>12}{'eff. perm.':>12}{'reference':>11}{'diff %':>8}")
for name, results, Z0, effective in lines:
    print(f"{name:<11}{results[-1]['Z0']:>11.2f}{Z0:>12.2f}{effective:>12.3f}{reference[name]:>11.2f}"
          f"{100 * (Z0 - reference[name]) / reference[name]:>8.2f}")

Machine used (Colab):

In [ ]:
!lscpu | grep "Model name"; nproc; free -h | grep Mem
import sys; print(sys.version)